In [ ]:
import sys
from pathlib import Path

import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import timm  # <-- Added timm for EfficientNetV2

from torch.utils.data import DataLoader

sys.path.append(
    str(
        Path("..")
        .resolve()
        / "src"
    )
)

from dataset_utils import EuroSATDataset
from training_utils import train_model

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
train_df = pd.read_csv(
    "../data/splits/train.csv"
)

validation_df = pd.read_csv(
    "../data/splits/validation.csv"
)

test_df = pd.read_csv(
    "../data/splits/test.csv"
)

with open(
    "../data/splits/metadata.json",
    "r"
) as file:

    metadata = json.load(
        file
    )

MEAN = metadata[
    "band_mean"
]

STD = metadata[
    "band_std"
]

print(
    len(train_df),
    len(validation_df),
    len(test_df)
)

print(
    MEAN
)

print(
    STD
)

In [ ]:
train_dataset = EuroSATDataset(
    train_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=True
)

validation_dataset = EuroSATDataset(
    validation_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=False
)

test_dataset = EuroSATDataset(
    test_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=False
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

In [ ]:
import timm

# Create EfficientNetV2 model adapted for 13 input channels and 10 classes
model = timm.create_model(
    'efficientnetv2_rw_m', 
    pretrained=True, 
    in_chans=13, 
    num_classes=10
)

model = model.to(device)

print("EfficientNetV2-S Model Loaded")
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)
print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

In [ ]:
criterion = nn.CrossEntropyLoss()

baseline_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        baseline_model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

baseline_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    baseline_optimizer,
    T_max=10
)

In [ ]:
Path("../models").mkdir(
    parents=True,
    exist_ok=True
)

baseline_model, baseline_history = train_model(
    model=baseline_model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=baseline_optimizer,
    scheduler=baseline_scheduler,
    device=device,
    epochs=10,
    patience=11,
    save_path="../models/efficientnetv2_frozen_best.pth",
    accumulation_steps=1
)

print("Frozen baseline training complete.")

In [ ]:
model = timm.create_model(
    'efficientnetv2_rw_m',
    pretrained=True,
    in_chans=13,
    num_classes=10
)

# Ensure all model layers are trainable.
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

print("Fully trainable EfficientNetV2")
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)
print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [ ]:
Path("../models").mkdir(
    parents=True,
    exist_ok=True
)

model, history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=10,
    patience=11,
    save_path="../models/efficientnetv2_best.pth",
    accumulation_steps=1
)

print("Full fine-tuning complete.")

In [ ]:
baseline_epochs = range(
    1,
    len(baseline_history["validation_accuracy"]) + 1
)

finetuning_epochs = range(
    1,
    len(history["validation_accuracy"]) + 1
)

# --------------------------------
# Validation Accuracy
# --------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    baseline_epochs,
    baseline_history["validation_accuracy"],
    marker="o",
    label="Frozen Baseline"
)

plt.plot(
    finetuning_epochs,
    history["validation_accuracy"],
    marker="o",
    label="Full Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("EfficientNetV2: Validation Accuracy Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

Path("../figures").mkdir(
    parents=True,
    exist_ok=True
)

plt.savefig(
    "../figures/efficientnetv2_validation_accuracy_comparison.png",
    dpi=300
)

plt.show()

# --------------------------------
# Validation Macro F1
# --------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(baseline_history["validation_macro_f1"]) + 1),
    baseline_history["validation_macro_f1"],
    marker="o",
    label="Frozen Baseline"
)

plt.plot(
    range(1, len(history["validation_macro_f1"]) + 1),
    history["validation_macro_f1"],
    marker="o",
    label="Full Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Macro F1")
plt.title("EfficientNetV2: Validation Macro F1 Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    "../figures/efficientnetv2_validation_macro_f1_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
experiments = {
    "Frozen Baseline": baseline_history,
    "Full Fine-Tuning": history
}

for name, results in experiments.items():

    best_accuracy = max(
        results["validation_accuracy"]
    )

    best_macro_f1 = max(
        results["validation_macro_f1"]
    )

    best_loss = min(
        results["validation_loss"]
    )

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Best validation accuracy: {best_accuracy:.4f}")
    print(f"Best validation macro F1: {best_macro_f1:.4f}")
    print(f"Lowest validation loss:   {best_loss:.4f}")